# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrahmanshaheen1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

# Read the private token from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Check the Colab Secrets panel."
    )

# Connect DuckDB to the protected Hugging Face dataset.
con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Feature month: information available by the end of March.
MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

# Outcome month: what happens after the decision moment.
APRIL_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("Warehouse connection configured successfully.")
print("Feature window: March 2026")
print("Outcome window: April 2026")

Warehouse connection configured successfully.
Feature window: March 2026
Outcome window: April 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis and time window

- **Lane:** Refresh / Content Opportunity Scoring.
- **Source table:** `fact_content_daily_performance`.
- **Raw table grain:** One row represents one webpage or content item for one client on one report date.
- **Final feature-frame grain:** One row represents one webpage or content item for one client.
- **Feature window:** March 1–31, 2026.
- **Decision moment:** The end of March 2026.
- **Outcome window:** April 1–30, 2026.
- **What I will rank:** Pages will be ranked by their risk of experiencing an impressions decline in April compared with March.
- **Provisional label:** `1` when April impressions are more than 20% below March impressions; otherwise `0`.
- **Human action:** The highest-ranked pages can be reviewed for refreshing, expanding, protecting, pruning, or monitoring.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

#### Features — exactly five

1. **`march_impressions`**  
   Knowable at the decision moment because it uses only impressions observed from March 1–31, 2026.

2. **`march_clicks`**  
   Knowable at the decision moment because it uses only clicks observed during March 2026.

3. **`march_ctr_pct`**  
   Knowable at the decision moment because it is calculated only from March clicks and March impressions.

4. **`march_avg_position`**  
   Knowable at the decision moment because it summarizes search position values observed during March.

5. **`march_active_days`**  
   Knowable at the decision moment because it counts March days on which the page received at least one impression.

#### Label / proxy

- **`is_next_month_decline`** is the provisional label.
- It equals `1` when April impressions are more than 20% lower than March impressions.
- It equals `0` otherwise.
- This is a future observed outcome used as a proxy for which pages may deserve review.

#### Context fields

- **`client_hash_id`** identifies the pseudonymized client and will be used for grouping and validation.
- **`content_hash_id`** identifies the pseudonymized webpage or content item.
- These identifiers are context only and will not be model features.

#### Excluded fields

- April measurements are excluded from the honest feature set because they occur after the March decision moment.
- `future_impression_ratio` is excluded because it directly reveals how the label was constructed.
- GA4 behavior columns are excluded from this first feature frame because GA4 coverage is incomplete for some clients and periods.
- Client and content identifiers are excluded from model inputs because their numeric or text codes have no meaningful predictive interpretation.
- Raw names, URLs, queries, and private client information are not used.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# Verification Query 1:
# Are there any duplicate rows at the documented
# report_date × client × content grain?

grain_check = con.sql(f"""
    SELECT COUNT(*) AS duplicate_grain_groups
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS rows_at_grain
        FROM {MARCH_DAILY}
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) > 1
    ) AS duplicates
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_grain_groups
0,0


In [3]:
# Verification Query 2:
# How large is the March slice, and which dates does it contain?

slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        MIN(report_date) AS minimum_date,
        MAX(report_date) AS maximum_date
    FROM {MARCH_DAILY}
""").df()

slice_check

,march_row_count,client_count,content_count,minimum_date,maximum_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [4]:
# Verification Query 3:
# How many March rows survive when GA4 data is genuinely available?

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS rows_with_ga4_available,

        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            ) /
            NULLIF(COUNT(*), 0),
            2
        ) AS percent_with_ga4_available

    FROM {MARCH_DAILY}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,rows_with_ga4_available,percent_with_ga4_available
0,9841378,413966,4.21


### Five-feature page-level frame

The following query converts the March daily table into one row per client and webpage. It then joins April totals only to construct the future decline label. April measurements are retained as label-source information and are not honest model features.

In [5]:
# Feature construction:
# March creates the five honest features.
# April is used only to create the future outcome label.

feature_frame = con.sql(f"""
    WITH march_features AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                COALESCE(gsc_impressions, 0)
            ) AS march_impressions,

            SUM(
                COALESCE(gsc_clicks, 0)
            ) AS march_clicks,

            ROUND(
                100.0 *
                SUM(COALESCE(gsc_clicks, 0)) /
                NULLIF(
                    SUM(COALESCE(gsc_impressions, 0)),
                    0
                ),
                4
            ) AS march_ctr_pct,

            AVG(
                CASE
                    WHEN gsc_impressions > 0
                         AND gsc_avg_position > 0
                    THEN gsc_avg_position
                END
            ) AS march_avg_position,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_impressions > 0
                    THEN report_date
                END
            ) AS march_active_days

        FROM {MARCH_DAILY}
        GROUP BY
            client_hash_id,
            content_hash_id

        -- Minimum evidence filter to reduce extremely sparse pages.
        HAVING SUM(COALESCE(gsc_impressions, 0)) >= 100
    ),

    april_outcomes AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                COALESCE(gsc_impressions, 0)
            ) AS label_source_april_impressions,

            COUNT(DISTINCT report_date) AS april_observed_days

        FROM {APRIL_DAILY}
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,

        -- Exactly five honest features
        m.march_impressions,
        m.march_clicks,
        m.march_ctr_pct,
        m.march_avg_position,
        m.march_active_days,

        -- Label source, not a feature
        a.label_source_april_impressions,

        CASE
            WHEN a.label_source_april_impressions
                 < 0.80 * m.march_impressions
            THEN 1
            ELSE 0
        END AS is_next_month_decline

    FROM march_features AS m
    INNER JOIN april_outcomes AS a
        USING (client_hash_id, content_hash_id)

    WHERE a.april_observed_days > 0
""").df()

FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_days",
]

print("Final unit of analysis: one row per client and webpage.")
print(f"Feature-frame rows: {len(feature_frame):,}")
print(f"Number of honest features: {len(FEATURES)}")
print(
    "Declining-label rate: "
    f"{feature_frame['is_next_month_decline'].mean():.1%}"
)

feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *FEATURES,
        "is_next_month_decline",
    ]
].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final unit of analysis: one row per client and webpage.
Feature-frame rows: 101,441
Number of honest features: 5
Declining-label rate: 51.7%


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_active_days,is_next_month_decline
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,0.0000,5.331238,29,1
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,0.1112,5.908100,31,0
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,0.0000,6.969536,30,1
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,0.0000,5.177774,31,1
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,0.1295,4.685335,31,1


### Deliberate leakage experiment

First, I train a quick model using only the five March features. Next, I deliberately add `future_impression_ratio`, which uses April outcome information and therefore leaks the answer. I compare the scores, delete the leaked column, and retain the honest result.

In [6]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline


def precision_at_k(y_true, scores, k=50):
    """Share of true declining pages among the highest k scores."""
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    actual_k = min(k, len(y_array))
    top_indices = np.argsort(score_array)[-actual_k:]

    return y_array[top_indices].mean()


model_frame = feature_frame.copy()

# Hold out entire clients rather than mixing pages
# from the same client across train and test.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_indices, test_indices = next(
    splitter.split(
        model_frame,
        groups=model_frame["client_hash_id"],
    )
)

train_df = model_frame.iloc[train_indices].copy()
test_df = model_frame.iloc[test_indices].copy()

y_train = train_df["is_next_month_decline"]
y_test = test_df["is_next_month_decline"]


def make_model():
    return Pipeline(
        steps=[
            (
                "missing_values",
                SimpleImputer(strategy="median"),
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    min_samples_leaf=10,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )


# ---------------------------
# Honest model
# ---------------------------

honest_model = make_model()

honest_model.fit(
    train_df[FEATURES],
    y_train,
)

honest_scores = honest_model.predict_proba(
    test_df[FEATURES]
)[:, 1]

honest_precision_50 = precision_at_k(
    y_test,
    honest_scores,
    k=50,
)


# ---------------------------
# Deliberate leakage
# ---------------------------

model_frame["future_impression_ratio"] = (
    model_frame["label_source_april_impressions"]
    / model_frame["march_impressions"]
)

LEAKED_FEATURES = FEATURES + [
    "future_impression_ratio"
]

leaked_train_df = model_frame.iloc[train_indices]
leaked_test_df = model_frame.iloc[test_indices]

leaked_model = make_model()

leaked_model.fit(
    leaked_train_df[LEAKED_FEATURES],
    y_train,
)

leaked_scores = leaked_model.predict_proba(
    leaked_test_df[LEAKED_FEATURES]
)[:, 1]

leaked_precision_50 = precision_at_k(
    y_test,
    leaked_scores,
    k=50,
)


print(
    f"Honest Precision@50: "
    f"{honest_precision_50:.3f}"
)

print(
    f"Leaked Precision@50: "
    f"{leaked_precision_50:.3f}"
)

print(
    "\nThe leaked score is invalid because "
    "future_impression_ratio uses April information."
)


# Delete the leaked column and keep only the honest feature design.
model_frame.drop(
    columns=["future_impression_ratio"],
    inplace=True,
)

assert "future_impression_ratio" not in model_frame.columns
assert "future_impression_ratio" not in FEATURES

print("\nLeak removed successfully.")
print("Final honest features:")
for feature in FEATURES:
    print(f"- {feature}")

print(
    f"\nFinal score retained for this notebook: "
    f"Honest Precision@50 = {honest_precision_50:.3f}"
)

Honest Precision@50: 0.620
Leaked Precision@50: 1.000

The leaked score is invalid because future_impression_ratio uses April information.

Leak removed successfully.
Final honest features:
- march_impressions
- march_clicks
- march_ctr_pct
- march_avg_position
- march_active_days

Final score retained for this notebook: Honest Precision@50 = 0.620


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This slice has several limitations.

First, the warehouse is an unbalanced panel: different clients began tracking at different dates, so not every client has the same amount of history. The March-to-April frame also includes only pages observed in both months and pages with at least 100 March impressions. Therefore, its findings may not apply to new pages, deleted pages, very-low-volume pages, or clients with limited tracking history.

Second, GA4 availability is incomplete for some clients and dates. Values recorded before GA4 tracking began cannot be interpreted as genuine zero engagement, which is why GA4 behavior metrics were excluded from this first five-feature frame.

March contains 31 days while April contains 30 days, so comparing raw monthly impression totals introduces a small calendar-length difference. A later version should compare normalized daily impression rates.

Finally, this data can show measured associations and support a review queue, but it cannot prove that refreshing a page causes impressions to recover. A human must still inspect each recommended page before taking action.

## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔ ] No client names, URLs, or private queries anywhere
- [ ✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.